# Test Symmetric Dirichlet Parametrization using MeshFEM Variant Settings

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark, flip_avoiding_step_length
import energy

import numpy as np
import time, copy
import param_utils
import helper_funcs
from helper_funcs import HessianStats
import igl

In [ ]:
import psutil

In [ ]:
import parallelism
parallelism.set_max_num_tbb_threads(psutil.cpu_count(logical=True))

## User Settings

In [ ]:
ProjectionStrategy = 'Always' # Adaptive
EigenvalueModification = 'Clamp' # Abs
ProjectionType = 'FBased' # XBased
AutodiffSetting = 'NoAD' # AD
SteplengthComputer = 'FlipAvoid' # NoFlipAvoid

In [ ]:
import MeshFEMParamSolverEnum
MeshFEMParamSolverEnum.optionNamesFromIndex(17)

In [ ]:
usr_hessian_shift = 1e-10

In [ ]:
SAVE_UV = False

## Read Mesh and Opt

In [ ]:
# m = mesh.Mesh('../../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh')
# m = mesh.Mesh('../../3rdparty/MeshFEM/misc/examples/meshes/uv-100.obj')
# m = mesh.Mesh('../../../Models/TableOneModels/cow2Disc.off')
mesh_file_path = '../../../Models/TableOneModels/Superman_cut3.off'
m = helper_funcs.read_mesh(mesh_file_path)

In [ ]:
model_name = os.path.splitext(os.path.basename(mesh_file_path))[0]

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
# Tutte Initialization
uv_init = parametrization.harmonic(m, bdry_uv)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)

uv.setVars(uv_init.ravel())

In [ ]:
obj_history = []
time_histroy = []
grad_norm_history = []

hessian_projected_history = []
hessian_shifted_amount_history = []

step_norm_history = []
directional_derivative_history = []


def customCallback(prob, cb_ind):
    if cb_ind > 1:
        hessian_projected_history.append(int(prob.hessianWasProjected))
        hessian_shifted_amount_history.append(prob.lastFactorizationShiftMagnitude)
    it_time = time.perf_counter()
    obj_history.append(prob.energy())
    time_histroy.append(it_time)
    grad_norm_history.append(np.linalg.norm(prob.gradient()))

def customSaveUVCallback(prob, cb_ind):
    if cb_ind > 1:
        hessian_projected_history.append(int(prob.hessianWasProjected))
        hessian_shifted_amount_history.append(prob.lastFactorizationShiftMagnitude)
    obj_history.append(prob.energy())
    grad_norm_history.append(np.linalg.norm(prob.gradient()))
    uv_fn = 'uv_ravel_'+ 'iter_' + str(cb_ind-1)
    uv_arr = uv.getVars()
    np.savez_compressed(os.path.join(result_path, uv_fn), arr=uv_arr)
    
def customSaveStepDCallback(prob, step, directional_derivative):
    step_norm_history.append(np.linalg.norm(step))
    directional_derivative_history.append(-directional_derivative)

In [ ]:
# Configuring User Options 
# AutodiffSetting
if AutodiffSetting == 'AD':  symmdiri_energy = energy.SymmetricDirichletDerivativeFree(2)
elif AutodiffSetting == 'NoAD': symmdiri_energy = energy.SymmetricDirichlet(2)
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Autodiff Setting {AutodiffSetting} is not implemented.")

# EigenvalueModification
if EigenvalueModification == 'Clamp': symmdiri_energy.useAbsProjection = False
elif EigenvalueModification == 'Abs': symmdiri_energy.useAbsProjection = True
else: raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Eigenvalue Modification {EigenvalueModification} is not implemented.")

# Construct `SymmetricDirichlet` parametrization energy and problem
param = mesh_energy.Parametrization(m, uv, symmdiri_energy)
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])

# Step Length Computer
if SteplengthComputer == 'FlipAvoid':
    prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
    prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8    # in accordance to Composite Majorization
elif SteplengthComputer == 'NoFlipAvoid':
    pass
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Steplength Computer {SteplengthComputer} is not implemented.")

if SAVE_UV: 
    uvsave_path = os.path.join('SAVE_UV', model_name)
    os.makedirs(uvsave_path, exist_ok=True)
    prob.setCustomIterationCallback(customSaveUVCallback)
else:
    prob.setCustomIterationCallback(customCallback)
prob.setCustomLineSearchBeganCallback(customSaveStepDCallback)

# Projection Type
if ProjectionType == 'FBased':  param.useXBasedProjection = False
elif ProjectionType == 'XBased': param.useXBasedProjection = True
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Projection Type {ProjectionType} is not implemented.")

In [ ]:
print("Initialization Done, Energy: ", prob.energy())

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
# prob.setCustomIterationCallback(v.updater())
v.show()

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = usr_hessian_shift
opt = prob.optimizer()

In [ ]:
opt.options.verboseNonPosDef = True
opt.options.niter = 200
# Projection Strategy
if ProjectionStrategy == 'Adaptive':
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
    opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
elif ProjectionStrategy == 'Always':
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
elif ProjectionStrategy == 'Never':
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Projection Strategy {ProjectionStrategy} is not implemented.")

In [ ]:
benchmark.reset()
start_time = time.time()
cr = opt.optimize()
benchmark.report()

In [ ]:
v.update()

In [ ]:
hessian_projected_history.append(int(prob.hessianWasProjected)) # The projection status of the Hessian used in are i-1
hessian_shifted_amount_history.append(prob.lastFactorizationShiftMagnitude)
# Also saving hessian_related information (arrays)
hessian_projected_arr = np.array(hessian_projected_history, dtype=int)
hessian_shifted_arr = np.array(hessian_shifted_amount_history, dtype=float)
hessian_indef_arr = np.array(cr.indefinite, dtype=int)

In [ ]:
if SAVE_UV:
    obj_arr = np.array(obj_history)
    grad_norm_arr = np.array(grad_norm_history)
    # we saved uv coordinates per-iteration and hessian_projected_history
    step_size_arr = np.array(step_norm_history)
    dd_arr = np.array(directional_derivative_history)

    obj_filename = 'obj_history.npy'
    grad_norm_filename = 'grad_norm_history.npy'
    hp_filename = 'hessian_projected_history.npy'
    hs_filename = 'hessian_shifted_amount_history.npy'
    hindef_filename = 'hessian_indefinite_history.npy'
    step_filename = 'step_size_history.npy'
    dd_filename = 'directional_derivative_history.npy'

    np.save(os.path.join(uvsave_path, obj_filename), obj_arr)
    np.save(os.path.join(uvsave_path, grad_norm_filename), grad_norm_arr)
    np.save(os.path.join(uvsave_path, hp_filename), hessian_projected_arr)
    np.save(os.path.join(uvsave_path, hs_filename), hessian_shifted_arr)
    np.save(os.path.join(uvsave_path, hindef_filename), hessian_indef_arr)
    np.save(os.path.join(uvsave_path, step_filename), step_size_arr)
    np.save(os.path.join(uvsave_path, dd_filename), dd_arr)
    print(f"[File] Saved UV '.npz' files, {obj_filename}, {grad_norm_filename}, {hp_filename}, {hs_filename}, {hindef_filename}, {step_filename}, {dd_filename} in {uvsave_path}.")

## Plot Convergency Plot

In [ ]:
import matplotlib
from matplotlib import pyplot as plt

In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(grad_norm_history)

plt.xlabel('Iter', fontsize=12)
plt.ylabel('Grad Norm', fontsize=12)
plt.title(f"Model: {model_name}", fontsize=10)
plt.yscale("log")

plt.grid(linestyle='--', linewidth=0.5)
plt.ylim(bottom=0)